In [20]:
from src.preprocessing import generate_diff
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("iberu/RunBugRun")

diff_generated = []
labels_generated = []

excluted_v1_classes = ['literal', 'function', 'variable_access', 'io', 'try_catch']

for row in ds['train']:
    if row['buggy_code'] == None or row['fixed_code'] == None:
        continue

    if row['labels'] is None or len(row['labels']) != 1:
        continue

    top_level_label = row['labels'][0].split('.')[0]

    if top_level_label in excluted_v1_classes:
        continue
    diff_generated.append(generate_diff(row['buggy_code'], row['fixed_code']))
    labels_generated.append(top_level_label)

In [ ]:
from collections import Counter
print('Printing out the first 5 examples from diff:')
for i in range(5):
    print(diff_generated[i])

print('Printing out the first 5 labels:')
for i in range(5):
    print(labels_generated[i])
    
print(f'Total generated diffs: {len(diff_generated)}')
print(f"Total training examples: {len(ds['train'])}")
print(f'Usable examples: {len(diff_generated)}')
print(f"Filtered out: {len(ds['train']) - len(diff_generated)}")

label_counts = Counter(labels_generated)
print(label_counts)

Printing out the first 5 examples from diff:
-         print(i,"x",j,"=",i*j)
+         print(i,"x",j,"=",i*j,sep="")
- [print('{}x{}={}').format(i, j, i * j) for j in range(1, 10) for i in range(1, 10)]
+ [print('{}x{}={}'.format(i, j, i * j)) for i in range(1, 10) for j in range(1, 10)]
- for i in range(10):
+ for i in range(1, 10):
-     for j in range(10):
+     for j in range(1, 10):
- 
- 		print(i,"x",j,"=",i*j,)
+ 		print(i,"x",j,"=",i*j,sep="")
+ #coding: UTF-8
+ 
- for i in range(10):
+ for i in range(1, 10):
- 	for j in range(10):
+ 	for j in range(1, 10):
Printing out the first 5 labels:
call
call
call
call
call
Total generated diffs: 35641
Total training examples: 133705
Usable examples: 35641
Filtered out: 98064
Counter({'call': 17165, 'expression': 7415, 'control_flow': 5394, 'assignment': 4586, 'identifier': 1081})


In [24]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter

X, y = diff_generated, labels_generated
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = .2, random_state = 42, stratify=y)
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'Counter for y_train: {Counter(y_train)}')
print(f'Counter for y_test: {Counter(y_test)}')




X_train shape: (28512, 5793)
X_test shape: (7129, 5793)
Counter for y_train: Counter({'call': 13731, 'expression': 5932, 'control_flow': 4315, 'assignment': 3669, 'identifier': 865})
Counter for y_test: Counter({'call': 3434, 'expression': 1483, 'control_flow': 1079, 'assignment': 917, 'identifier': 216})
